# Topic 7: Sorting Algorithms

**Goal**: Understand how sorting works under the hood, implement the major algorithms, know when to use each.  
**Time**: ~5-6 hours  
**Prereqs**: Topics 0-2, 6

---

## Why Learn Sorting?

Sorting is fundamental — many problems become trivial once data is sorted (binary search, merge intervals, two pointers on sorted data). Plus, sorting algorithms teach core techniques: divide-and-conquer, partitioning, and stability.

| Algorithm | Time (Best) | Time (Avg) | Time (Worst) | Space | Stable? |
|---|---|---|---|---|---|
| Bubble Sort | O(n) | O(n²) | O(n²) | O(1) | Yes |
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) | No |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | Yes |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) | No |
| Counting Sort | O(n+k) | O(n+k) | O(n+k) | O(k) | Yes |
| Python's sort | O(n) | O(n log n) | O(n log n) | O(n) | Yes |

---

## Bubble Sort

**Idea**: Walk through the array, compare adjacent pairs, swap if out of order. After one full pass, the largest element "bubbles up" to the end. Repeat for the remaining unsorted portion.

**Optimization**: If a full pass makes zero swaps, the array is already sorted — stop early.

```
Array: [5, 3, 8, 1, 2]

Pass 1 — compare adjacent pairs, swap if needed:

  [5, 3, 8, 1, 2]    5 > 3? Yes → swap
   ^  ^
  [3, 5, 8, 1, 2]    5 > 8? No
      ^  ^
  [3, 5, 8, 1, 2]    8 > 1? Yes → swap
         ^  ^
  [3, 5, 1, 8, 2]    8 > 2? Yes → swap
            ^  ^
  [3, 5, 1, 2, 8]    8 is now in its final position ✓
                ^

Pass 2 — only look at [3, 5, 1, 2] (8 is done):

  [3, 5, 1, 2, 8]    3 > 5? No
   ^  ^
  [3, 5, 1, 2, 8]    5 > 1? Yes → swap
      ^  ^
  [3, 1, 5, 2, 8]    5 > 2? Yes → swap
         ^  ^
  [3, 1, 2, 5, 8]    5 is now in place ✓
            ^

...continue until no swaps needed.
```

Each pass pushes the next-largest element into place. Like bubbles rising to the surface.

In [ ]:
def bubble_sort(arr):
    arr = arr[:]
    n = len(arr)

    for i in range(n):
        swapped = False

        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True

        print(f"  Pass {i + 1}: {arr}  (sorted from right: {arr[n - i:]})")  # noqa

        if not swapped:
            print("  No swaps — already sorted, stopping early!")
            break

    return arr


print("Bubble Sort Trace:")
print(f"  Input:  {[5, 3, 8, 1, 2]}")
result = bubble_sort([5, 3, 8, 1, 2])
print(f"  Result: {result}")

print()
print("Already sorted (early stop):")
print(f"  Input:  {[1, 2, 3, 4, 5]}")
result = bubble_sort([1, 2, 3, 4, 5])
print(f"  Result: {result}")

---

## Selection Sort

**Idea**: Find the minimum element in the unsorted portion, swap it into the next sorted position. Repeat.

```
Array: [29, 10, 14, 37, 13]

Step 1: Find min in [29, 10, 14, 37, 13] → 10 at index 1
        Swap arr[0] and arr[1]
        [10, 29, 14, 37, 13]
         ✓

Step 2: Find min in [29, 14, 37, 13] → 13 at index 4
        Swap arr[1] and arr[4]
        [10, 13, 14, 37, 29]
         ✓   ✓

Step 3: Find min in [14, 37, 29] → 14 at index 2
        Already in place!
        [10, 13, 14, 37, 29]
         ✓   ✓   ✓

Step 4: Find min in [37, 29] → 29 at index 4
        Swap arr[3] and arr[4]
        [10, 13, 14, 29, 37]
         ✓   ✓   ✓   ✓   ✓
```

**Why not stable?** Consider `[5a, 3, 5b]`. We find min=3, swap it with 5a → `[3, 5a, 5b]`... wait, that's fine here. But `[5a, 5b, 3]` → find min=3, swap with 5a → `[3, 5b, 5a]`. Now 5a comes after 5b — original order of equal elements was broken.

In [ ]:
def selection_sort(arr):
    arr = arr[:]
    n = len(arr)

    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            if arr[j] < arr[min_idx]:
                min_idx = j

        arr[i], arr[min_idx] = arr[min_idx], arr[i]

        sorted_part = arr[:i + 1]
        unsorted_part = arr[i + 1:]
        print(f"  Step {i + 1}: min={arr[i]}, placed at index {i}  →  {sorted_part} | {unsorted_part}")

    return arr


print("Selection Sort Trace:")
print(f"  Input: {[29, 10, 14, 37, 13]}")
result = selection_sort([29, 10, 14, 37, 13])
print(f"  Result: {result}")

---

## Insertion Sort

**Idea**: Like sorting a hand of cards. You pick up one card at a time and insert it into the correct position among the cards already in your hand.

The array has a **sorted boundary** that moves from left to right. Everything to the left of the boundary is sorted.

```
Array: [5, 2, 4, 6, 1, 3]

Start: sorted | unsorted
       [5]    | [2, 4, 6, 1, 3]

Take 2: Insert into sorted portion
       [5] ← 2 goes before 5
       [2, 5] | [4, 6, 1, 3]

Take 4: Insert into sorted portion
       [2, 5] ← 4 goes between 2 and 5
       [2, 4, 5] | [6, 1, 3]

Take 6: Insert into sorted portion
       [2, 4, 5] ← 6 goes at end (already in place)
       [2, 4, 5, 6] | [1, 3]

Take 1: Insert into sorted portion
       [2, 4, 5, 6] ← 1 goes at beginning
       [1, 2, 4, 5, 6] | [3]

Take 3: Insert into sorted portion
       [1, 2, 4, 5, 6] ← 3 goes between 2 and 4
       [1, 2, 3, 4, 5, 6] | []
```

**Why O(n) best case?** If the array is already sorted, each element is already in place — the inner loop never executes. This makes insertion sort great for nearly-sorted data.

In [ ]:
def insertion_sort(arr):
    arr = arr[:]
    n = len(arr)

    for i in range(1, n):
        key = arr[i]
        j = i - 1

        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1

        arr[j + 1] = key

        sorted_part = arr[:i + 1]
        unsorted_part = arr[i + 1:]
        marker = f"inserted {key} at index {j + 1}"
        print(f"  Step {i}: {marker:<25} →  {sorted_part} | {unsorted_part}")

    return arr


print("Insertion Sort Trace:")
print(f"  Input: {[5, 2, 4, 6, 1, 3]}")
result = insertion_sort([5, 2, 4, 6, 1, 3])
print(f"  Result: {result}")

---

## Merge Sort

THE most important sort to understand. It introduces **divide-and-conquer**:
1. **Divide**: Split the array in half
2. **Conquer**: Recursively sort each half
3. **Combine**: Merge the two sorted halves

The key insight: merging two sorted arrays into one sorted array is O(n).

### Full Recursion Tree

```
                   [38, 27, 43, 3, 9, 82, 10]
                         /              \
              [38, 27, 43, 3]        [9, 82, 10]
                /        \             /      \
           [38, 27]    [43, 3]     [9, 82]   [10]
            /   \       /   \       /   \      |
          [38] [27]   [43]  [3]   [9]  [82]  [10]
            \   /       \   /       \   /      |
           [27, 38]   [3, 43]     [9, 82]    [10]
               \        /             \       /
           [3, 27, 38, 43]        [9, 10, 82]
                    \                /
            [3, 9, 10, 27, 38, 43, 82]
```

Notice:
- **Top half**: splitting (log n levels of splitting)
- **Bottom half**: merging (log n levels of merging)
- Each level does O(n) total work merging → O(n log n) total

### The Merge Step — In Detail

Merging two sorted arrays is the heart of merge sort. Use two pointers, one for each array. Compare, take the smaller, advance that pointer.

```
Merge [3, 27, 38] and [9, 10, 82]:

  left:  [3, 27, 38]    right: [9, 10, 82]    result: []
          ^                      ^
          i                      j

  3 < 9  → take 3               result: [3]
  left:  [3, 27, 38]    right: [9, 10, 82]
              ^                  ^

  27 > 9 → take 9               result: [3, 9]
  left:  [3, 27, 38]    right: [9, 10, 82]
              ^                      ^

  27 > 10 → take 10             result: [3, 9, 10]
  left:  [3, 27, 38]    right: [9, 10, 82]
              ^                          ^

  27 < 82 → take 27             result: [3, 9, 10, 27]
  left:  [3, 27, 38]    right: [9, 10, 82]
                  ^                      ^

  38 < 82 → take 38             result: [3, 9, 10, 27, 38]
  left exhausted → append remaining right
                                 result: [3, 9, 10, 27, 38, 82]  ✓
```

In [ ]:
def merge(left, right):
    """Merge two sorted arrays into one sorted array."""
    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result


print("Merge two sorted arrays:")
left = [3, 27, 38]
right = [9, 10, 82]
print(f"  Left:   {left}")
print(f"  Right:  {right}")
print(f"  Merged: {merge(left, right)}")

In [ ]:
def merge_sort(arr, depth=0):
    indent = "    " * depth

    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    print(f"{indent}Split: {arr} → {arr[:mid]} + {arr[mid:]}")

    left = merge_sort(arr[:mid], depth + 1)
    right = merge_sort(arr[mid:], depth + 1)

    merged = merge(left, right)
    print(f"{indent}Merge: {left} + {right} → {merged}")
    return merged


print("Merge Sort — Full Trace:")
data = [38, 27, 43, 3, 9, 82, 10]
print(f"Input: {data}")
print()
result = merge_sort(data)
print(f"\nResult: {result}")

---

## Quick Sort

**Idea**: Pick a **pivot** element. Partition the array so that everything less than the pivot goes left, everything greater goes right. The pivot lands in its final sorted position. Recurse on left and right.

```
pivot = 4
[3, 6, 8, 10, 1, 2, 4]
 ↓ partition
[3, 1, 2] [4] [6, 8, 10]
  smaller  pivot  larger
```

### Partition Step (Lomuto scheme — pivot = last element)

```
Array: [3, 6, 8, 10, 1, 2, 4]     pivot = 4 (last element)

i = boundary of "≤ pivot" region (starts before array)
j scans left to right:

  j=0: arr[0]=3  ≤ 4? Yes → swap arr[0] with arr[0], i=0
       [3, 6, 8, 10, 1, 2, 4]
        i

  j=1: arr[1]=6  ≤ 4? No → skip
       [3, 6, 8, 10, 1, 2, 4]
        i

  j=2: arr[2]=8  ≤ 4? No → skip

  j=3: arr[3]=10 ≤ 4? No → skip

  j=4: arr[4]=1  ≤ 4? Yes → i=1, swap arr[1] with arr[4]
       [3, 1, 8, 10, 6, 2, 4]
           i

  j=5: arr[5]=2  ≤ 4? Yes → i=2, swap arr[2] with arr[5]
       [3, 1, 2, 10, 6, 8, 4]
              i

  Done scanning. Place pivot at i+1:
  Swap arr[3] with arr[6] (pivot)
       [3, 1, 2, 4, 6, 8, 10]
                 ^ pivot in final position!

  Result: [3, 1, 2]  [4]  [6, 8, 10]
           ≤ pivot   pivot  > pivot
```

In [ ]:
def partition(arr, low, high):
    pivot = arr[high]
    i = low - 1

    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]

    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1


def quick_sort(arr, low, high, depth=0):
    if low < high:
        indent = "    " * depth
        sub = arr[low:high + 1]
        pi = partition(arr, low, high)

        left_part = arr[low:pi]
        right_part = arr[pi + 1:high + 1]
        print(f"{indent}Partition {sub} → {left_part} [{arr[pi]}] {right_part}")

        quick_sort(arr, low, pi - 1, depth + 1)
        quick_sort(arr, pi + 1, high, depth + 1)


print("Quick Sort — Full Trace:")
data = [3, 6, 8, 10, 1, 2, 4]
print(f"Input: {data}")
print()
quick_sort(data, 0, len(data) - 1)
print(f"\nResult: {data}")

### Pivot Choice Matters

The choice of pivot determines performance:

```
GOOD pivot (near median)         BAD pivot (min or max element)
Splits roughly in half           One side is empty!

      [........]                       [........]
      /        \                       /        \
  [....] p [....]                  [] p [.......]
  /   \   /   \                        /       \
 [..][..][..][..]                   [] p [......]
                                       /       \
 depth = log n                      [] p [.....]
 total = O(n log n)                    ...n levels
                                   total = O(n²)
```

**Strategies to pick a good pivot**:

| Strategy | How | Worst-case avoidance |
|---|---|---|
| First/Last element | `pivot = arr[high]` | Terrible on sorted data |
| Random | `pivot = arr[random index]` | Very unlikely to hit O(n²) |
| Median-of-three | Median of first, middle, last | Good balance, low overhead |

**Quick sort vs Merge sort**:
- Quick sort is in-place (O(log n) stack space), merge sort needs O(n) extra space
- Quick sort has O(n²) worst case, merge sort is always O(n log n)
- In practice, quick sort is often faster due to cache locality

---

## Counting Sort

**Idea**: Don't compare elements at all! Count how many times each value appears, then reconstruct the sorted array from the counts. Only works when the range of values (k) is small.

```
Array: [4, 2, 2, 8, 3, 3, 1]

Step 1: Count occurrences (range 0..8)

  Value:  0  1  2  3  4  5  6  7  8
  Count: [0, 1, 2, 2, 1, 0, 0, 0, 1]
              ^  ^  ^  ^           ^
              1× 2× 2× 1×         1×

Step 2: Reconstruct from counts

  1 appears 1 time  → [1]
  2 appears 2 times → [1, 2, 2]
  3 appears 2 times → [1, 2, 2, 3, 3]
  4 appears 1 time  → [1, 2, 2, 3, 3, 4]
  8 appears 1 time  → [1, 2, 2, 3, 3, 4, 8]
```

**Time**: O(n + k) where k = range of values  
**Space**: O(k) for the count array  
**When to use**: Values are integers in a known, small range (e.g., ages 0-150, grades 0-100, ASCII chars 0-127)

In [ ]:
def counting_sort(arr):
    if not arr:
        return arr

    min_val, max_val = min(arr), max(arr)
    range_size = max_val - min_val + 1
    count = [0] * range_size

    for num in arr:
        count[num - min_val] += 1

    print(f"  Count array (values {min_val}..{max_val}):")
    for i, c in enumerate(count):
        if c > 0:
            print(f"    value {i + min_val}: {'█' * c} ({c}×)")

    result = []
    for i, c in enumerate(count):
        result.extend([i + min_val] * c)

    return result


print("Counting Sort Trace:")
data = [4, 2, 2, 8, 3, 3, 1]
print(f"  Input:  {data}")
print()
result = counting_sort(data)
print(f"\n  Result: {result}")

---

## Python's Built-in Sort — Timsort

Python uses **Timsort**, a hybrid of merge sort and insertion sort designed for real-world data.

How it works:
1. Find naturally occurring sorted subsequences ("runs") in the data
2. Extend short runs using insertion sort (great for small/nearly-sorted chunks)
3. Merge runs together using merge sort logic

**Two ways to sort**:

| | `sorted(iterable)` | `list.sort()` |
|---|---|---|
| Returns | New sorted list | `None` (sorts in-place) |
| Original | Unchanged | Modified |
| Works on | Any iterable | Only lists |

**Key features**:
- **key function**: Transform elements before comparison (doesn't change the actual data)
- **reverse**: Sort descending
- **Stable**: Equal elements keep their original relative order

In [ ]:
# sorted() vs .sort()
original = [3, 1, 4, 1, 5, 9, 2, 6]

new_list = sorted(original)
print(f"sorted() returns new list: {new_list}")
print(f"original unchanged:        {original}")

original.sort()
print(f"after .sort(), original:   {original}")

print()

# Key functions
words = ["banana", "pie", "Washington", "a"]
print(f"Sort by length:     {sorted(words, key=len)}")
print(f"Sort alphabetical:  {sorted(words, key=str.lower)}")

print()

# Sort complex objects
students = [
    ("Alice", 85),
    ("Bob", 92),
    ("Charlie", 85),
    ("Diana", 92),
]
by_grade = sorted(students, key=lambda s: s[1])
print(f"By grade:  {by_grade}")

by_grade_desc = sorted(students, key=lambda s: s[1], reverse=True)
print(f"By grade (desc): {by_grade_desc}")

print()

# Stability proof: sort by grade, equal grades keep original name order
print("Stability proof:")
print(f"  Original order: {[s[0] for s in students]}")
print(f"  Sorted by grade: {by_grade}")
print(f"  Alice (85) still before Charlie (85) ✓  — stable!")
print(f"  Bob (92) still before Diana (92) ✓  — stable!")

print()

# Multi-key sort: sort by grade (desc), then by name (asc) within same grade
multi_key = sorted(students, key=lambda s: (-s[1], s[0]))
print(f"By grade desc, then name asc: {multi_key}")

---

## Stability Explained

A sort is **stable** if elements with equal keys keep their original relative order.

**Why does this matter?** Imagine sorting a spreadsheet:

```
Original data (already sorted by name):

  Name       Grade
  ─────────  ─────
  Alice      B
  Bob        A
  Charlie    B
  Diana      A
  Eve        B

Now sort by Grade using a STABLE sort:

  Name       Grade
  ─────────  ─────
  Bob        A       ← A's keep their relative order (Bob before Diana)
  Diana      A
  Alice      B       ← B's keep their relative order (Alice, Charlie, Eve)
  Charlie    B
  Eve        B

Sort by Grade using an UNSTABLE sort:

  Name       Grade
  ─────────  ─────
  Diana      A       ← A's might be reordered! (Diana before Bob)
  Bob        A
  Eve        B       ← B's might be shuffled! (Eve, Alice, Charlie)
  Alice      B
  Charlie    B
```

**Rule of thumb**: When sorting by multiple criteria sequentially (sort by name, then by grade), you need a stable sort so the first sort's order is preserved within ties.

---

## When to Use Which?

```
                    Need to sort?
                         |
              ┌──────────┴──────────┐
              │                     │
        Small n (< ~50)?      Large n?
              │                     │
       Insertion Sort        ┌──────┴──────┐
      (low overhead)         │             │
                       Need stable?   Don't care?
                             │             │
                        Merge Sort    Quick Sort
                       (guaranteed     (faster in
                        O(n log n))    practice)

  Special cases:
  ─────────────
  • Small integer range    → Counting Sort O(n+k)
  • Nearly sorted data     → Insertion Sort O(n) best case
  • Linked list            → Merge Sort (no random access needed)
  • In production Python   → sorted() / .sort() (Timsort)
```

### Quick Reference

| Situation | Best Choice | Why |
|---|---|---|
| General purpose, any input | Merge Sort | O(n log n) guaranteed |
| General purpose, avg case speed | Quick Sort | Cache-friendly, low overhead |
| Nearly sorted data | Insertion Sort | O(n) best case |
| Small arrays (< ~50 elements) | Insertion Sort | Low constant factors |
| Integer values in small range | Counting Sort | O(n+k), beats comparison sorts |
| Need stability | Merge Sort / Timsort | Preserves relative order |
| Writing production Python | `sorted()` / `.sort()` | Timsort — optimized hybrid |

---

## Practice Problems

| # | Problem | LC # | Key Idea | Difficulty |
|---|---------|------|----------|------------|
| 1 | Sort Colors | 75 | Dutch National Flag / partition (not comparison sort) | Medium |
| 2 | Merge Intervals | 56 | Sort by start, then merge overlapping | Medium |
| 3 | Kth Largest Element in an Array | 215 | Quick Select (partition-based) | Medium |
| 4 | Sort List | 148 | Merge sort on linked list | Medium |
| 5 | Largest Number | 179 | Custom comparator — sort strings by concatenation | Medium |
| 6 | Meeting Rooms | 252 | Sort by start time, check overlaps | Easy |
| 7 | Valid Anagram | 242 | Sort both strings and compare (or use counting) | Easy |
| 8 | Relative Sort Array | 1122 | Counting sort idea with custom order | Easy |
| 9 | Sort Array By Parity | 905 | Two-pointer partition | Easy |
| 10 | Wiggle Sort II | 324 | Median finding + rearrangement | Medium |

---

## Pattern Cheat Sheet

```
┌──────────────────────────────────────────────────────────────────┐
│                    SORTING PATTERNS                             │
├──────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CUSTOM COMPARATOR                                              │
│    sorted(items, key=lambda x: ...)                             │
│    Sort by transformed value without changing data.             │
│    Example: sort strings by length, tuples by second element.   │
│                                                                 │
│  SORT + TWO POINTERS                                           │
│    Sort first, then use two pointers to find pairs/triplets.    │
│    Example: 3Sum, two sum on sorted array.                      │
│                                                                 │
│  SORT + MERGE INTERVALS                                        │
│    Sort intervals by start time, then merge overlapping ones.   │
│    Example: merge intervals, meeting rooms.                     │
│                                                                 │
│  PARTITION (Quick Select)                                      │
│    Use partitioning to find kth element in O(n) average.        │
│    Example: kth largest, sort colors.                           │
│                                                                 │
│  COUNTING / BUCKET SORT                                        │
│    When values have small range or you need frequency info.     │
│    Example: top k frequent, sort colors, relative sort.         │
│                                                                 │
│  STABILITY TRICK                                               │
│    Sort by secondary key first, then primary key (stable).      │
│    Or use tuple keys: key=lambda x: (primary, secondary).       │
│                                                                 │
└──────────────────────────────────────────────────────────────────┘
```

---

**Next up: Topic 8 — Searching Algorithms**